In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
from model import FM_PhysMamba_UNET
from losses import FM_PhysicalLoss
from torch.optim import Adam
from tqdm.notebook import tqdm

In [2]:
version_1  = FM_PhysMamba_UNET(model_cfg_path = "small", use_version = 1)
version_2  = FM_PhysMamba_UNET(model_cfg_path = "small", use_version = 2)

In [3]:
print(version_2)

FM_PhysMamba_UNET(
  (time_mlp): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): SiLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
  )
  (phys_gate): Sequential(
    (0): Linear(in_features=259, out_features=256, bias=True)
    (1): SiLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
  )
  (down_time_projs): ModuleList(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): Linear(in_features=256, out_features=256, bias=True)
    (2): Linear(in_features=256, out_features=512, bias=True)
  )
  (up_time_projs): ModuleList(
    (0): Linear(in_features=256, out_features=512, bias=True)
    (1): Linear(in_features=256, out_features=256, bias=True)
    (2): Linear(in_features=256, out_features=128, bias=True)
  )
  (init_conv): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (downs): ModuleList(
    (0): ModuleList(
      (0): PhysConvNeXtBlock(
        (dwconv): Conv2d(64, 64, kernel_siz

### Loading the Dataset

In [4]:
from data.utils import get_haze_transforms
from data import OHAZE_Dataset, DENSE_Haze_Dataset
from torch.utils.data import Subset, DataLoader

In [5]:
train_transform = get_haze_transforms(
    dataset_name = "OHAZE",
    resize_size = 256,
    split = "train",
    verbose = True
)


╭────────────────────────────────────────── Augmentation Pipeline ──────────────────────────────────────────╮
│ OHAZE | TRAIN | 256x256                                                                                   │
│ ├── 1. Geometric (Synchronous)                                                                            │
│ │   ├── Applies identically to CLEAR & HAZY for alignment                                                 │
│ │   └── Compose(                                                                                          │
│ │             RandomCrop(size=(256, 256), pad_if_needed=True, fill=0, padding_mode=constant)              │
│ │             RandomHorizontalFlip(p=0.5)                                                                 │
│ │             RandomVerticalFlip(p=0.5)                                                                   │
│ │       )                                                                                                 │
│ ├── 2. Appearance (Hazy-Only)                                                                             │
│ │   ├── Simulates real-world haze variations                                                              │
│ │   └── Compose(    ColorJitter(brightness=(0.95, 1.05), contrast=(0.95, 1.05), saturation=(0.95, 1.05))) │
│ └── 3. Common (Tensor & Norm)                                                                             │
│     ├── Final prep: ToTensor, Normalize                                                                   │
│     └── Compose(                                                                                          │
│               Resize(size=[256, 256], interpolation=InterpolationMode.BILINEAR, antialias=True)           │
│               ToImage()                                                                                   │
│               ToDtype(scale=True)                                                                         │
│               Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5], inplace=False)                         │
│         )                                                                                                 │
╰───────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [6]:
sanity_transform = get_haze_transforms(
    dataset_name = "DENSEHAZE",
    resize_size = 256,
    split = "sanity",
    verbose = True
)


Transform Mode: SANITY (Deterministic Resize)


In [7]:
sanity_dataset = DENSE_Haze_Dataset(
    root_dir = 'dataset/dense-haze', 
    transform = sanity_transform
)

# 2. Select specific indices (e.g., the first 8 images)
# You can also pick specific "hard" indices if you know them (e.g., [4, 12, 15...])
indices = list(range(8))

# 3. Create the Subset
# This creates a "view" of the dataset that only contains those 8 images
tiny_subset = Subset(sanity_dataset, indices)

# 4. Create the DataLoader
# We use a batch_size of 4, so this loader will run for exactly 2 steps per epoch.
sanity_loader = DataLoader(
    tiny_subset,
    batch_size=4,
    shuffle=False,  # Shuffle is good to check if the model is robust
    num_workers=4
)

In [8]:
# --- VERIFICATION STEP ---
# Run this once to make sure shapes are correct [B, C, H, W]
print(f"Subset created with {len(tiny_subset)} images.")

for batch_idx, data in enumerate(sanity_loader):
    # Depending on your dataset return format, data might be (hazy, clean) or a dict
    # Assuming standard tuple (hazy, clean):
    if isinstance(data, (list, tuple)):
        hazy, clean = data
        print(f"Batch {batch_idx}: Input Shape {hazy.shape}, Target Shape {clean.shape}")
    else:
        print(f"Batch {batch_idx}: Data received")
        
    if batch_idx >= 1: break # Just check one or two batches

Subset created with 8 images.
Batch 0: Input Shape torch.Size([4, 3, 256, 256]), Target Shape torch.Size([4, 3, 256, 256])
Batch 1: Input Shape torch.Size([4, 3, 256, 256]), Target Shape torch.Size([4, 3, 256, 256])


### Try Sanity Test

In [9]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TOTAL_STEPS = 600

criterion = FM_PhysicalLoss().to(DEVICE)
# criterion = torch.nn.SmoothL1Loss(beta=1.0)
model_2  = FM_PhysMamba_UNET(model_cfg_path = "small", use_version = 2).to(DEVICE)
optimizer = Adam(model_2.parameters(), lr=5e-4)


# ==========================================
# 4. THE OVERFIT LOOP (500 Steps)
# ==========================================
print("Starting Overfit Training...")
model_2.train()

iterator = iter(sanity_loader)

pbar = tqdm(range(TOTAL_STEPS), desc="Overfitting", dynamic_ncols=True)

for step in pbar:
    # 1. Get Batch (Cycle infinitely)
    try:
        clean, hazy = next(iterator)
    except StopIteration:
        iterator = iter(sanity_loader)
        clean, hazy = next(iterator)
    hazy, clean = hazy.to(DEVICE), clean.to(DEVICE)
    
    # Fake timestep for testing (or random)
    # t = torch.randint(0, 1000, (hazy.shape[0],), device=DEVICE).long()
    t = torch.rand((clean.shape[0],), device=DEVICE)
    
    # Forward Pass
    # Note: Your forward returns (v_pred, t_map, A_pred)
    v_pred, t_map, A_pred = model_2(hazy, t)
    
    # Calculate target v (for Flow Matching)
    # v_t = clean - hazy (Simple Flow Matching target)
    target_v = clean - hazy 
    
    # Calculate Loss
    loss, loss_dict = criterion(
        (v_pred, t_map, A_pred), 
        target_v, 
        hazy, # x_t (using hazy as noisy input for this test)
        t, 
        clean, 
        hazy
    )
    # loss = criterion(v_pred, target_v)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # 7. UPDATE TQDM BAR
    # Formatting exactly as you requested
    status_str = (
        f"L:{loss.item():.3f} | "
        f"F:{loss_dict['Flow']:.3f} "
        f"P:{loss_dict['Phys']:.3f} "
        f"FFT:{loss_dict['FFT']:.3f} "
        f"V:{loss_dict['VGG']:.3f}"
    )
    # status_str = f"Loss (SmoothL1): {loss.item():.6f}"
    pbar.set_description(status_str)

    # 8. Success Condition
    if loss.item() < 0.002:
        pbar.write("✅ Converged! Loss is near zero. Architecture works.")
        break

pbar.close()

Starting Overfit Training...


Overfitting:   0%|          | 0/600 [00:00<?, ?it/s]

In [11]:
import torchvision.utils as vutils
import os

# Create a debug folder
os.makedirs("debug_images", exist_ok=True)

iterator = iter(sanity_loader)
clean, hazy = next(iterator)
hazy, clean = hazy.to(DEVICE), clean.to(DEVICE)

t = torch.rand((clean.shape[0],), device=DEVICE)

# 1. Get the last batch processed
with torch.no_grad():
    # Predict again on the current batch
    v_pred, t_map, A_pred = model_2(hazy, t)
    
    # Reconstruct the Clean Image J_pred
    # Flow Matching: clean = hazy + v * (1 - 0) approx
    # Since we are essentially predicting (clean - hazy), 
    # J_pred = hazy + v_pred
    J_pred = hazy + v_pred

# 2. Concatenate for easy comparison: [Hazy | Prediction | Clean]
# We clamp to ensure valid image range [0,1]
comparison = torch.cat([
    hazy[:4],       # Input
    J_pred[:4],     # Your Model's Output
    clean[:4]       # Ground Truth
], dim=0)

# 3. Save
save_path = "debug_images/sanity_result_v3.png"
vutils.save_image(comparison, save_path, nrow=4, normalize=True)

print(f"📸 Saved visual check to {save_path}")
print("Check this image. If the 'Middle Row' looks somewhat like the 'Bottom Row', START TRAINING.")

📸 Saved visual check to debug_images/sanity_result_v3.png
Check this image. If the 'Middle Row' looks somewhat like the 'Bottom Row', START TRAINING.


In [12]:
print(A_pred.min())
print(A_pred.max())

tensor(0.3611, device='cuda:0')
tensor(0.9748, device='cuda:0')


In [13]:
vutils.save_image(t_map[:4], "debug_images/t_map_check_v3.png", normalize=True)


In [14]:
def analyze_physics_sanity(t_map, A_pred, save_dir="debug_images"):
    # 1. Analyze Transmission Map (t_map)
    # Check if t_map is 'flat' (bad) or has 'structure' (good)
    t_min, t_max = t_map.min().item(), t_map.max().item()
    t_mean = t_map.mean().item()
    
    # 2. Analyze Atmosphere (A_pred)
    # Convert from [-1, 1] back to [0, 1] for easier reading
    A_val = A_pred.mean(dim=(0, 2, 3)).cpu().numpy()

    print("\n--- 🔎 Physics Head Sanity Report ---")
    print(f"Transmission -> Range: [{t_min:.3f}, {t_max:.3f}] | Mean: {t_mean:.3f}")
    print(f"Atmosphere   -> RGB: [{A_val[0]:.3f}, {A_val[1]:.3f}, {A_val[2]:.3f}]")

    if t_max - t_min < 0.01:
        print("⚠️ WARNING: t_map is very flat. The model might be ignoring depth.")
    else:
        print("✅ SUCCESS: t_map has structural variety.")

    if A_val.mean() < 0.2:
        print("⚠️ WARNING: A_pred is very dark. Atmosphere should typically be bright.")
    else:
        print("✅ SUCCESS: A_pred looks physically plausible.")

# Run the analysis
analyze_physics_sanity(t_map, A_pred)


--- 🔎 Physics Head Sanity Report ---
Transmission -> Range: [0.031, 0.830] | Mean: 0.548
Atmosphere   -> RGB: [0.610, 0.668, 0.832]
✅ SUCCESS: t_map has structural variety.
✅ SUCCESS: A_pred looks physically plausible.


In [17]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TOTAL_STEPS = 600

criterion = FM_PhysicalLoss().to(DEVICE)
# criterion = torch.nn.SmoothL1Loss(beta=1.0)
model_1  = FM_PhysMamba_UNET(model_cfg_path = "small", use_version = 1).to(DEVICE)
optimizer = Adam(model_1.parameters(), lr=5e-4)


# ==========================================
# 4. THE OVERFIT LOOP (500 Steps)
# ==========================================
print("Starting Overfit Training...")
model_1.train()

iterator = iter(sanity_loader)

pbar = tqdm(range(TOTAL_STEPS), desc="Overfitting", dynamic_ncols=True)

for step in pbar:
    # 1. Get Batch (Cycle infinitely)
    try:
        clean, hazy = next(iterator)
    except StopIteration:
        iterator = iter(sanity_loader)
        clean, hazy = next(iterator)
    hazy, clean = hazy.to(DEVICE), clean.to(DEVICE)
    
    # Fake timestep for testing (or random)
    # t = torch.randint(0, 1000, (hazy.shape[0],), device=DEVICE).long()
    t = torch.rand((clean.shape[0],), device=DEVICE)
    
    # Forward Pass
    # Note: Your forward returns (v_pred, t_map, A_pred)
    v_pred, t_map, A_pred = model_1(hazy, t)
    
    # Calculate target v (for Flow Matching)
    # v_t = clean - hazy (Simple Flow Matching target)
    target_v = clean - hazy 
    
    # Calculate Loss
    loss, loss_dict = criterion(
        (v_pred, t_map, A_pred), 
        target_v, 
        hazy, # x_t (using hazy as noisy input for this test)
        t, 
        clean, 
        hazy
    )
    # loss = criterion(v_pred, target_v)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # 7. UPDATE TQDM BAR
    # Formatting exactly as you requested
    status_str = (
        f"L:{loss.item():.3f} | "
        f"F:{loss_dict['Flow']:.3f} "
        f"P:{loss_dict['Phys']:.3f} "
        f"FFT:{loss_dict['FFT']:.3f} "
        f"V:{loss_dict['VGG']:.3f}"
    )
    # status_str = f"Loss (SmoothL1): {loss.item():.6f}"
    pbar.set_description(status_str)

    # 8. Success Condition
    if loss.item() < 0.002:
        pbar.write("✅ Converged! Loss is near zero. Architecture works.")
        break

pbar.close()

Starting Overfit Training...


Overfitting:   0%|          | 0/600 [00:00<?, ?it/s]

In [18]:
import torchvision.utils as vutils
import os

# Create a debug folder
os.makedirs("debug_images", exist_ok=True)

iterator = iter(sanity_loader)
clean, hazy = next(iterator)
hazy, clean = hazy.to(DEVICE), clean.to(DEVICE)

t = torch.rand((clean.shape[0],), device=DEVICE)

# 1. Get the last batch processed
with torch.no_grad():
    # Predict again on the current batch
    v_pred, t_map, A_pred = model_1(hazy, t)
    
    # Reconstruct the Clean Image J_pred
    # Flow Matching: clean = hazy + v * (1 - 0) approx
    # Since we are essentially predicting (clean - hazy), 
    # J_pred = hazy + v_pred
    J_pred = hazy + v_pred

# 2. Concatenate for easy comparison: [Hazy | Prediction | Clean]
# We clamp to ensure valid image range [0,1]
comparison = torch.cat([
    hazy[:4],       # Input
    J_pred[:4],     # Your Model's Output
    clean[:4]       # Ground Truth
], dim=0)

# 3. Save
save_path = "debug_images/sanity_result_v2_new.png"
vutils.save_image(comparison, save_path, nrow=4, normalize=True)

print(f"📸 Saved visual check to {save_path}")
print("Check this image. If the 'Middle Row' looks somewhat like the 'Bottom Row', START TRAINING.")

📸 Saved visual check to debug_images/sanity_result_v2_new.png
Check this image. If the 'Middle Row' looks somewhat like the 'Bottom Row', START TRAINING.


In [19]:
vutils.save_image(t_map[:4], "debug_images/t_map_check_v2_new.png", normalize=True)
